In [1]:
import eikon
import pandas
import datetime
import matplotlib.pyplot as plt
import time
import pathlib

ModuleNotFoundError: No module named 'eikon'

## Linking table

In [ ]:
df_links = pandas.read_csv('processed/linking_table/linking_table.csv', index_col=0, dtype=str)

In [ ]:
df_links.head(2)

## Fetch data from Eikon

In [ ]:
eikon.set_app_key('592a8e92287443d0aac94c27bbd6a8d8d7c81482')

Data Item Browser

In [ ]:
# From the Data Item Browser
field_price_usd = eikon.TR_Field(field_name='TR.CLOSEPRICE',
                                 params={
                                     'Curn': 'USD',
                                     'Adjusted': '1',
                                 })
field_date = eikon.TR_Field(field_name='TR.CLOSEPRICE.date')

In [ ]:

while True:
    try:
        ts_downloaded = [path.stem for path in pathlib.Path('raw/eikon/time_series/price_2009_2023/').rglob('*.csv')]
        ts_downloaded[:2]

        for reuters_id in df_links['reuters_id'].dropna().unique():
            if reuters_id in ts_downloaded:
                print(reuters_id, 'is already downloaded.')
                continue
            time.sleep(2)
            print('Downloading:', reuters_id)
            df, err = eikon.get_data(instruments=[reuters_id],
                                    fields=[field_price_usd, field_date],
                                    parameters={'SDate': '2009-01-01',
                                                'EDate': '2023-05-01',
                                                'Frq': 'D',
                                                },
                                    field_name=True
                                    )
            df.to_csv(f'raw/eikon/time_series/price_2009_2023/{reuters_id}.csv')

        break
    except eikon.EikonError as err:
        print(str(err.message))
        if err.code == 400:
            print('Bad request. Backend error. Restarting.')
            time.sleep(20)
        elif err.code == 504:
            print('Server timeout.')
            time.sleep(20)
        else:
            break


In [ ]:

while True:
    try:
        ts_downloaded = [path.stem for path in pathlib.Path('raw/eikon/time_series/esg_2007_2023/').rglob('*.csv')]

        for reuters_id in df_links['reuters_id'].dropna().unique():
            if reuters_id in ts_downloaded:
                print(reuters_id, 'is already downloaded.')
                continue

            time.sleep(2)
            
            print('Downloading:', reuters_id)
            
            df, err = eikon.get_data(instruments=[reuters_id],
                                    fields=[
                                        eikon.TR_Field(field_name='TR.TRESGScore.value'),
                                        eikon.TR_Field(field_name='TR.TRESGScore.date'),

                                        eikon.TR_Field(field_name='TR.EnvironmentPillarScore.value'),
                                        eikon.TR_Field(field_name='TR.EnvironmentPillarScore.date'),

                                        eikon.TR_Field(field_name='TR.SocialPillarScore.value'),
                                        eikon.TR_Field(field_name='TR.SocialPillarScore.date'),

                                        eikon.TR_Field(field_name='TR.GovernancePillarScore.value'),
                                        eikon.TR_Field(field_name='TR.GovernancePillarScore.date'),
                                    ],
                                    parameters={'SDate': -15,
                                                'EDate': 1,
                                                'Frq': 'FY',
                                                'IncludePartialYear': True,
                                                },
                                    field_name=True
                                    )

            df.to_csv(f'raw/eikon/time_series/esg_2007_2023/{reuters_id}.csv')

        break
    except eikon.EikonError as err:
        print(str(err.message))
        if err.code == 400:
            print('Bad request. Backend error. Restarting.')
            time.sleep(20)
        elif err.code == 504:
            print('Server timeout.')
            time.sleep(20)
        else:
            break


